# 01 · Un LLM local desde Python

El modelo de lenguaje corre en esta máquina, servido por Ollama. Ollama habla el mismo protocolo que la API de OpenAI, así que se usa con el SDK `openai` de Python apuntado a la dirección local: el SDK es solo el cliente, el modelo es Qwen y nada sale a internet.

Qué vamos a ver:

1. Qué modelos hay en el servidor.
2. Una pregunta y su respuesta.
3. La respuesta en streaming, con tokens por segundo.
4. Muchas peticiones a la vez: cuántos tokens por segundo da la máquina en total.

In [ ]:
import os

from openai import OpenAI

# La dirección del servidor y el modelo vienen del archivo .env del repositorio.
BASE_URL = os.environ["LLM_BASE_URL"]
API_KEY = os.environ.get("LLM_API_KEY", "local")  # Ollama no la revisa, pero el SDK pide una
MODEL = os.environ["LLM_MODEL"]

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

print("Servidor:", BASE_URL)
print("Modelo:  ", MODEL)

## 1. Qué modelos hay en el servidor

Conforme terminan las descargas, aquí aparecen los modelos nuevos. Para usar otro, basta cambiar la variable `MODEL`.

In [ ]:
for model in client.models.list():
    print(model.id)

## 2. Una pregunta, una respuesta

Un mensaje `system` le dice al modelo cómo comportarse; el mensaje `user` es la pregunta.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Eres asistente de una planta de manufactura. Responde en español, en pocas líneas."},
        {"role": "user", "content": "¿Qué es el OEE y para qué sirve?"},
    ],
)

print(response.choices[0].message.content)
print()
print("Tokens de entrada:", response.usage.prompt_tokens, "| tokens generados:", response.usage.completion_tokens)

## 3. Streaming

Con `stream=True` el servidor manda cada token en cuanto lo genera. Así se miden las dos cifras que muestra la interfaz web: cuánto tarda el primer token y cuántos tokens salen por segundo.

In [ ]:
import time


def stream_answer(prompt, model=MODEL):
    start = time.perf_counter()
    first_token_at = None
    usage = None

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True},  # el último mensaje trae el conteo de tokens
    )
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
        if not chunk.choices:
            continue
        piece = chunk.choices[0].delta.content
        if piece:
            if first_token_at is None:
                first_token_at = time.perf_counter()
            print(piece, end="", flush=True)

    end = time.perf_counter()
    first_token_at = first_token_at or end
    print("\n")
    print(f"Primer token: {(first_token_at - start) * 1000:.0f} ms")
    if usage and end > first_token_at:
        print(f"Velocidad:    {usage.completion_tokens / (end - first_token_at):.1f} tokens/s ({usage.completion_tokens} tokens)")


stream_answer("Explica en un párrafo qué es el mantenimiento predictivo.")

## 4. Muchas peticiones a la vez

Un chat usa la GPU un rato y la deja libre. Lo que distingue a una máquina como esta es atender a muchos usuarios al mismo tiempo: cada respuesta sale un poco más lenta, pero en total se generan muchos más tokens por segundo.

Mandamos la misma petición 1, 4, 8 y 16 veces en paralelo y medimos el total. El número de peticiones que el servidor atiende a la vez lo fija `OLLAMA_NUM_PARALLEL` en `.env`; las que pasan de ese número esperan su turno.

In [ ]:
import asyncio

import pandas as pd
from openai import AsyncOpenAI

async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

PROMPT = "Escribe un párrafo de unas 80 palabras sobre la importancia del mantenimiento preventivo."
MAX_TOKENS = 96  # todas las respuestas del mismo largo, para que la comparación sea pareja
LEVELS = [1, 4, 8, 16]


async def one_request():
    response = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=MAX_TOKENS,
    )
    return response.usage.completion_tokens


async def measure(parallel):
    start = time.perf_counter()
    tokens = await asyncio.gather(*[one_request() for _ in range(parallel)])
    seconds = time.perf_counter() - start
    return {
        "peticiones en paralelo": parallel,
        "segundos": seconds,
        "tokens generados": sum(tokens),
        "tokens/s en total": sum(tokens) / seconds,
    }


await one_request()  # una de calentamiento, para no medir la carga del modelo

results = pd.DataFrame([await measure(parallel) for parallel in LEVELS])
results.round(1)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.bar(results["peticiones en paralelo"].astype(str), results["tokens/s en total"], width=0.3, color="#059669")
ax.bar_label(bars, labels=[f"{value:.0f}" for value in results["tokens/s en total"]], padding=4, fontsize=12)
ax.set_title("Tokens por segundo en total", loc="left", fontsize=13, fontweight="bold")
ax.set_xlabel("Peticiones en paralelo")
ax.set_yticks([])
ax.margins(y=0.2)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
plt.show()

base = results["tokens/s en total"].iloc[0]
best = results.loc[results["tokens/s en total"].idxmax()]
gain = best["tokens/s en total"] / base
if gain >= 1.2:
    print(f"Con {int(best['peticiones en paralelo'])} peticiones en paralelo: {gain:.1f} veces los tokens por segundo de una sola.")
else:
    print("El total no sube con más peticiones: el servidor las está atendiendo de una en una.")
    print("Revisa OLLAMA_NUM_PARALLEL en .env.")

Cómo leer la gráfica: la barra crece mientras el servidor todavía puede atender más peticiones a la vez, y se aplana cuando llega a su límite (`OLLAMA_NUM_PARALLEL`). Si todas las barras salen parecidas, el servidor está atendiendo una sola petición a la vez.

Para repetirlo con otro modelo, cambia `MODEL` en la segunda celda por uno de la lista del paso 1 y vuelve a ejecutar.